# Stage C3 — Hyperparameter Optimization

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.5.1 (Table 8), feedback item suggesting
Optuna over a grid search.

**The Table 8 vs. Sec. 3.5.1 gap, resolved by using Optuna:** the prose
in 3.5.1 describes a 3-factor grid (learning rate x batch size x *one*
shared physics-weight scaling factor = 36 combinations), while Table 8
independently lists 13 dimensions, including **four separate** physics-
weight categories. A grid over all 13 isn't practical; that's exactly
why the feedback suggested Optuna's TPE sampler instead of a grid —
this notebook searches the full Table 8 space directly, so the two
descriptions don't need to be reconciled by hand.

**A second gap, not previously flagged:** Table 8 includes `ReLU` as a
candidate activation. C1's HC-λ convexity constraint (Eq. 3.11) needs
a **second** derivative — ReLU is piecewise-linear, so its second
derivative is zero almost everywhere. Trials that sample ReLU will
show a near-zero HC-λ penalty *by construction*, not because the
network actually learned a convex HC-λ relationship. This isn't fixed
here — Table 8 says ReLU is a candidate, so it stays a candidate — but
if the search picks ReLU as best, check HC-λ convexity empirically
(plot predicted HC vs. λ) before trusting the loss value alone.

**Framework note:** the composite loss (C2) depends on collocation-
point *inputs*, not just training pairs, so `model.fit()` can't be used
directly — this notebook writes a small custom training loop
(`tf.GradientTape` around the weights, on top of C2's own tapes around
the inputs).

**Input:** `data/masters_data.xlsx`, `outputs/C1_collocation_points.csv`,
`outputs/C2_loss_config.json`
**Output:** best hyperparameter configuration
(`outputs/C3_best_config.json`) and the full trial history, used by C4.

**Runtime:** `N_TRIALS` x 5 folds x a full physics-informed training
run each — much slower per-fit than B1 (the composite loss needs
several nested gradient tapes per step). Defaults below are a small
smoke-test scale (`N_TRIALS=20`); raise it once you've confirmed the
notebook runs end-to-end.


## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from pathlib import Path
from sklearn.model_selection import KFold
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)
print("optuna    ", optuna.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## 1. Load data, collocation points, and C2's calibrated weights

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

cv_pool = df.filter(pl.col("split") != "test")
X_cv = cv_pool.select([f"{c}_norm" for c in INPUT_COLS]).to_numpy().astype(np.float32)
Y_cv = cv_pool.select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy().astype(np.float32)
print("CV pool:", X_cv.shape[0], "points (test held out)")

colloc_df = pl.read_csv(OUT_DIR / "C1_collocation_points.csv")
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
rho_colloc = colloc_df["density_ratio"].to_numpy()
validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()

with open(OUT_DIR / "C2_loss_config.json") as f:
    loss_config = json.load(f)
w_final_base = loss_config["w_final"]  # C2's calibrated values, scaled per-trial below
print("Loaded", X_colloc.shape[0], "collocation points and C2's base weights.")


## 2. Table 8 search space

13 dimensions. The four physics-weight hyperparameters are **scaling
factors on C2's calibrated $w_j^{\text{final}}$**, one shared per
constraint *category* (Sec. 3.5.1's prose uses this "scaling factor"
language for a single shared factor; Table 8 splits it into four —
implementation choice made explicit here, not fully pinned down by the
text).

In [ ]:
CONSTRAINT_CATEGORY = {
    "NOx-SOI": "monotonic", "PM-lambda": "monotonic",
    "HC-lambda": "shape",
    "NOx-PM": "tradeoff", "eta-NOx": "tradeoff",
    "non-negativity": "nonneg",
}

def suggest_hyperparameters(trial):
    return {
        # architectural
        "n_layers": trial.suggest_int("n_layers", 1, 4),
        "n_units": trial.suggest_categorical("n_units", [8, 16, 32, 64, 128]),
        "activation": trial.suggest_categorical("activation", ["tanh", "sigmoid", "relu"]),
        # optimization
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [8, 16, 32]),
        "beta_1": trial.suggest_float("beta_1", 0.85, 0.95),
        "beta_2": trial.suggest_float("beta_2", 0.99, 0.999),
        # physics weight category scales
        "scale_monotonic": trial.suggest_float("scale_monotonic", 0.01, 10.0, log=True),
        "scale_shape": trial.suggest_float("scale_shape", 0.01, 10.0, log=True),
        "scale_tradeoff": trial.suggest_float("scale_tradeoff", 0.001, 5.0, log=True),
        "scale_nonneg": trial.suggest_float("scale_nonneg", 0.1, 20.0, log=True),
        # regularization
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.0, 0.5),
    }

CATEGORY_KEY = {"monotonic": "scale_monotonic", "shape": "scale_shape",
                 "tradeoff": "scale_tradeoff", "nonneg": "scale_nonneg"}


## 3. Model builder (with dropout) and the six physics penalties

Same formulas as C1/C2, repeated here so this notebook is self-
contained; the only addition is a `Dropout` layer after each hidden
layer, since dropout rate is now a search dimension.

In [ ]:
def build_model(n_layers, n_units, activation, dropout_rate, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    model = keras.Sequential([keras.Input(shape=(input_dim,))])
    for _ in range(n_layers):
        model.add(layers.Dense(n_units, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(output_dim, activation="linear"))
    return model

def monotonic_decreasing_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, d))

def convexity_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, -d_second))

def tradeoff_per_point(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.square(tf.maximum(0.0, grad_a * grad_b))

def nonneg_loss(predict_fn, x, out_idxs=EMISSION_IDXS):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, -emissions)), axis=1))


## 4. Custom training step

`model.fit()` can't take collocation points as a second, differently-
shaped input alongside `(X, Y)`, so training is one explicit step:
an outer tape differentiates the (already tape-using) composite loss
with respect to the model's *weights*, then the optimizer applies
those gradients. Weight decay (Eq. 3.19's regularization weight, here
a per-trial hyperparameter instead of C2's fixed $10^{-5}$) is added
directly in the loss, not via the optimizer.

In [ ]:
def composite_loss(model, X, Y, X_colloc, rho_colloc, validity_eff_nox,
                    w_scaled, weight_decay, sigma2):
    predict_fn = lambda x: model(x, training=True)
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    Y_t = tf.convert_to_tensor(Y, dtype=tf.float32)
    pred = predict_fn(X_t)
    l_data = tf.reduce_mean(tf.reduce_sum(tf.square(pred - Y_t) / sigma2, axis=1))

    per_point = {
        "NOx-SOI": monotonic_decreasing_per_point(predict_fn, X_colloc, NOX_IDX, SOI_IDX),
        "PM-lambda": monotonic_decreasing_per_point(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX),
        "HC-lambda": convexity_per_point(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX),
        "NOx-PM": tradeoff_per_point(predict_fn, X_colloc, NOX_IDX, PM_IDX, SOI_IDX),
        "eta-NOx": tradeoff_per_point(predict_fn, X_colloc, ETA_IDX, NOX_IDX, SOI_IDX),
    }
    l_phys = 0.0
    for name, pp in per_point.items():
        validity = validity_eff_nox if name == "eta-NOx" else np.ones(len(X_colloc))
        lambda_x = tf.constant(rho_colloc * validity, dtype=tf.float32)
        l_phys += float(w_scaled[name]) * tf.reduce_mean(lambda_x * pp)
    l_phys += float(w_scaled["non-negativity"]) * nonneg_loss(predict_fn, X_colloc)

    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    l_reg = weight_decay * sq_sum / model.count_params()

    return l_data + l_phys + l_reg, l_data

def train_step(model, optimizer, *args):
    with tf.GradientTape() as tape:
        loss, l_data = composite_loss(model, *args)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, l_data


## 5. Objective function — 5-fold CV, mean validation MSE

Per Sec. 3.5.1: the optimization target is plain validation MSE (data
fit), not the physics-inclusive loss — physics still shapes *training*
through Section 4's composite loss, it just isn't what Optuna scores.
Short epoch/patience budget here (search phase, many trials); C4 gets
the full budget for the final ensemble.

In [ ]:
N_FOLDS = 5
EPOCHS_PER_TRIAL = 150
PATIENCE_PER_TRIAL = 20

def objective(trial):
    hp = suggest_hyperparameters(trial)
    w_scaled = {name: w_final_base[name] * hp[CATEGORY_KEY[CONSTRAINT_CATEGORY[name]]]
                for name in w_final_base}

    fold_mses = []
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold_i, (tr_idx, va_idx) in enumerate(kf.split(X_cv)):
        keras.backend.clear_session()
        Xtr, Ytr = X_cv[tr_idx], Y_cv[tr_idx]
        Xva, Yva = X_cv[va_idx], Y_cv[va_idx]
        sigma2 = tf.constant(np.maximum(Ytr.var(axis=0, ddof=1), 1e-8), dtype=tf.float32)

        model = build_model(hp["n_layers"], hp["n_units"], hp["activation"],
                             hp["dropout_rate"], seed=SEED * 100 + fold_i)
        optimizer = keras.optimizers.Adam(learning_rate=hp["learning_rate"],
                                           beta_1=hp["beta_1"], beta_2=hp["beta_2"])

        best_val, patience_ctr, best_weights = np.inf, 0, None
        n_batches = max(1, len(Xtr) // hp["batch_size"])
        for epoch in range(EPOCHS_PER_TRIAL):
            perm = np.random.permutation(len(Xtr))
            for b in range(n_batches):
                idx = perm[b * hp["batch_size"]:(b + 1) * hp["batch_size"]]
                if len(idx) == 0:
                    continue
                train_step(model, optimizer, Xtr[idx], Ytr[idx], X_colloc,
                           rho_colloc, validity_eff_nox, w_scaled, hp["weight_decay"], sigma2)
            val_pred = model(Xva, training=False).numpy()
            val_mse = float(np.mean((val_pred - Yva) ** 2))
            if val_mse < best_val - 1e-6:
                best_val, patience_ctr = val_mse, 0
                best_weights = model.get_weights()
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE_PER_TRIAL:
                    break
        if best_weights is not None:
            model.set_weights(best_weights)
        fold_mses.append(best_val)

    trial.set_user_attr("fold_mses", fold_mses)
    return float(np.mean(fold_mses))


## 6. Run the study

In [ ]:
N_TRIALS = 20  # smoke-test scale; raise once the notebook runs cleanly end-to-end

sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", sampler=sampler, study_name="C3_pinn_hparams")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial: #{study.best_trial.number}  value={study.best_value:.5f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k:16s} {v}")


## 7. Optimization history

In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(width=800, height=420)
fig.show()


## 8. Parallel coordinates — hyperparameters vs. objective

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(width=950, height=500)
fig.show()


## 9. Parameter importance

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.update_layout(width=800, height=450)
fig.show()


## 10. Best configuration — a closer look

Per-fold spread for the winning trial (not just its mean), and a
reminder to check HC-λ convexity by hand if `activation == "relu"` won.

In [ ]:
best_fold_mses = study.best_trial.user_attrs["fold_mses"]
print(f"Best trial fold MSEs: {[round(v, 5) for v in best_fold_mses]}")
print(f"mean={np.mean(best_fold_mses):.5f}  std={np.std(best_fold_mses, ddof=1):.5f}")

if study.best_params["activation"] == "relu":
    print("\nNOTE: best trial uses ReLU -- HC-lambda convexity (Eq. 3.11) will show near-zero")
    print("penalty regardless of whether the learned HC(lambda) shape is actually convex.")
    print("Plot predicted HC vs. lambda for this configuration before trusting that term.")


## Optional — persist outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
best_config = dict(study.best_params)
best_config["cv_mean_mse"] = study.best_value
best_config["cv_fold_mses"] = best_fold_mses
with open(OUT_DIR / "C3_best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

trials_df = study.trials_dataframe()
pl.from_pandas(trials_df).write_csv(OUT_DIR / "C3_trials_history.csv")
print(f"Saved to {OUT_DIR}")


## Next

**C4** trains the final 10-model ensemble using this configuration —
architecture, learning rate, batch size, dropout, and the scaled
physics weights — with the full epoch/patience budget (Adam, then
L-BFGS) that C3's search deliberately shortened.
